# FusionHacks2026 Challenge

A volumetric 14 MeV neutron source is a fusion device aimed at producing neutrons from fusion, rather than producing energy. Neutrons produced from such a device could be used for a variety of applications, including the production of medical isotopes, accurate testing of the neutron reslience of future fusion reactor components, and the production of tritium. For this hackathon, we narrow our scope to the design of a **volumetric neutron source for medical isotope production.**

Since neutrons are, of course, generated by fusion reactions, one would have the same heuristics in mind for designing a neutron source as when designing a fusion reactor. Improving the rate of neutron production inside of your reactor is precisely the same problem as increasing the rate of fusion reactions. The salient difference is that the triple product must be above a certain threshold in order for a fusion power plant to be commerically viable; the threshold for a viable volumetric neutron source would be a lot smaller (and determined by you in this project). 

In reality, some fusion startups are taking an approach where they seek to build a small neutron source machine to derisk technologies that they eventually wish to use in the context of a power plant. 

We challenge you to design a volumetric neutron source based off of a **QA stellarator design** (i.e. a quasi-axiysmmetric stellarator). The design you submit will consist of three components:
1. A **_vacuum_** MHD equilibrium,
2. A coilset,
3. A business case for the reactor. 

## Designing the equilibrium and coils 

Designing the equilibrium and coils is a multi-objective optimization problem: essentially, we are interested in increasing the rate of neutron production, and decreasing the costs. 


### The equilibrium

The rate of neutron production is given by the following equation:
$$R = \frac{n^2}{4}\langle \sigma v \rangle V$$
where $n$ is the number density of ions in the reactor, $\langle \sigma v \rangle(T)$ is the reaction rate parameter (a known function of the temperature $T$ that is essentially proportional to the likelihood of a fusion reaction occuring). To compute the reaction rate given a reactor geometry, we need to determine the **density** and the **temperature** in a self-consistent manner. 

In a real fusion reactor, density and temperature are functions of the minor-radial coordinate, and are determined by the energy and particle transport equations. Solving the transport equations accurately is a catastrophically complicated problem: it depends on the neoclassical and turbulent heat fluxes, the magnetic geometry, the heating scheme, the fueling scheme, etc. However, for the sake of this hackathon, we can take a step back from the details of the physics itself, and look at the power balance of a reactor:

$$P_{\text{ext}} + P_\alpha = P_{\text{loss}} = \frac{W}{\tau_E}$$

where $P_{\text{ext}}$ is the external heating power, $P_\alpha$ is the heating power due to fusion-born alpha particles, $W$ is the total energy in the plasma, and $\tau_E$ is the energy confinement time. We can sub in expressions $W = \frac{3}{2}nkT$ and $P_\alpha = \frac{n^2}{4}\langle \sigma v \rangle E_\alpha$ (note that the latter is exactly the same functional form as the rate of neutron production, just multiplied by $E_\alpha$, the energy of the fusion born alpha):

$$P_{\text{ext}} + \frac{n^2}{4}\langle \sigma v \rangle E_\alpha = \frac{\frac{3}{2}nkT}{\tau_E}$$

All of the complicated physics of the reactor and any gains we get from tuning the geometry of the reactor enter this power balance through the energy confinement time $\tau_E$. But how do we know exactly how our reactor geometry impacts this? The zeroth order thing one can do when attempting to make an engineering decision in the absence of detailed physical knowledge is to look at relevant experimental data to form a *scaling law*. Scalings are particularly a big deal in the fusion business, and are used to inform the design of future machines. 
The ISS04 scaling is given as:

$$\tau_{E,ISS04} = H0.465B^{0.84}R^{0.64}a^{2.28}n^{0.54}P_{\text{ext}}^{-0.61}\iota^{0.41}_{2/3}$$

where $B$ \[$\text{T}$\] is the magnetic field strength, $R$ \[$\text{m}$\] is the major radius, $a$ \[$\text{m}$\] is the minor radius, $n$ \[$10^{20} \text{m}^{-3}$\] is the number density, $P_{\text{ext}}$ \[$\text{MW}$\] is the heating power, and $\iota_{2/3}$ is the rotational transform at the $\rho=2/3$ surface. All of these quantities (save the $\iota_{2/3}$) are taken as averages. Pay close attention to the exponents in this equations. The $H$ factor is a quantity that is used to quantify the improvement of a reactor's performance over the scaling; i.e., experimentally, 

$$H = \frac{\tau_{E, \text{experimental}}}{\tau_{E,ISS04}}$$

It should be noted that many of the stellarators used in the ISS04 scaling were not quasi-symmetric and exhibited poor neoclassical transport. Neoclassically optimized (such as QA stellarators) will tend to have a higher H value. This number is usually within the range of $0.7-1.8$. We model $H$ as 

$$H=H_{min}+\frac{H_{max}-H_{min}}{1+\exp\left(k\left(\log{\frac{\epsilon}{\epsilon_0}}+\mu\right)\right)}$$

where $\epsilon$ is the average effective ripple (which is corrected with a reduction in neoclassical transport), and $\epsilon_0$ is a reference effective ripple, taken as $\epsilon_0=0.1$. We set $\mu=4$, $H_{min} = 0.7$, and $H_{max} = 2$. This is a logistic function in the log space of $\epsilon$: the idea is that it should motivate you to get an average effective ripple $\epsilon$ on the order of around $10^{-5}$ or $10^{-6}$, with diminishing returns after. Some modifications to the starter code below stellarator should be able to achieve this off the bat! 

Please note: **you should not directly optimize for $\epsilon$!** It is a much harder target function than the quasisymmetry metric `desc.objectives.QuasisymmetryTwoTerm`, which it is correlated to. The idea of optimizing a simpler target function as a proxy for another quantity is very common in stellarator and optimization in general. 

Returning to the problem of solving for the neutron rate, we need two more things to close the equation such that we can solve for T only: we need to specify the density $n$ and the external heating power $P_{\text{ext}}$. 

We will limit the density $n$ by enforcing a limit on the quantity $\beta$:
$$\beta=\frac{nkT}{B^2/2\mu_0}$$
which is the ratio of the plasma's fluid pressure and the device's "magnetic pressure", with all quantities in SI units. In most reactor studies today, this value is most optimistically taken at $5\%$, which we will follow. We can isolate this equation for temperature, and substitute it back into our power balance. 

Finally, we must specify the external heating power. This power must be chosen by you, and will be factored into the cost. A reasonable heating power will be on the order 10 MW. 

Altogther, this gives, for the power balance:

$$P_{\text{ext}} + \frac{n(T, B)^2}{4}\langle \sigma v \rangle(T) E_\alpha = \frac{\frac{3}{2}n(T, B)kT}{\tau_E(B, R, a, T, P_\text{ext}, \iota_{2/3})}$$

We implement a function in python below that solves for T in this equation given the rest of your quantities. Given this T, we can solve for the fusion reaction rate from your design, for which we also implement a python function.

> These functions are now implemented in `fusionhacks_metrics.py`; the temperature is `temp_from_eq(eq, P_ext):` and the neutron fluence (i.e., total rate of neutron production) is in `neutron_fluence(eq,T)`. 

Some tips on the optimization of the equilibrium
- Again, **do not optimize for $\epsilon$ directly** — use one of the quasisymmetry target functions from the DESC package instead
- We are optimizing the **vacuum equilibria** - we are not considering the coil currents and we are setting the pressure and current profiles to zero for our optimization
- You may be tempted to just use the neutron reaction rate itself as a target function. Besides the fact that this would involve optimizing $\epsilon$ directly, this is generally not a great idea, because **the same equilibrium geometry can be scaled to a different average field strength $B$ and a different size (different $a$ or $R$ or $V$) without changing the structure of the magnetic field**. What you should do instead is figure out, based on the ISS04 scaling law, some non-dimensional geometrical constraints (think aspect ratio, iota, elongation, number of fields periods, etc), and perform your optimization with these constraints (either implemented as constraints or as part of your target function, as in the example), and then when you're satisfied with your equilibrium, pick $B$, $a$, $R$, and $P_\text{ext}$ to make your cost-reaction rate trade-off. 

### The coils

The coils are optimized given an equilibrium, with the task of simply matching the equilbrium field at the boundary. That is, fundamentally, the coils must minimize the following

$$\frac{\boldsymbol{B}\cdot \hat{\boldsymbol{n}}}{B}|_{\rho=1}$$

which is the field stength-normalized projection of the magnetic field due to the coils on the boundary of the plasma. This quantity should be of order $10^{-3}$. In our scoring, if your coils have field error in excess of $5\cdot10^{-3}$, we will begin penalizing your reaction rate. The coils are important in determining the cost of your reactor, which will be computed via the following formula:

$$C = w_L L + w_V V^{1.2} + w_\kappa(\langle\kappa\rangle + w_{\kappa,m} |\kappa_{\text{max}}|)^{1.7} + w_{\text{heating}} P_\text{ext} $$

where $L$ is the total current-length of the coils \[$\text{A} \cdot \text{m}$\], $V$ is the equilibrium volume \[$\text{m}^3$\], and $\langle\kappa\rangle, |\kappa_{\text{max}}|$ are the root-mean-square and maximum coil curvatures, respectively. 

We also impose the following constraints:
- $\frac{\boldsymbol{B}\cdot \hat{\boldsymbol{n}}}{B}|_{\rho=1}$ should be around $5\cdot10^{-3}$. Otherwise, in our scoring, your reaction rate will be penalized. 
- plasma-coil distance should be at least 15% of the minor rarius
- coil-coil distance should be at least 5% of the minor radius

Some tips on optimizing for coils
- There are a couple ways of optimizing coils in DESC - check out the tutorials in the DESC documentation!

## The business case

A medical isotope that would benefit greatly from the production of fast neutrons is Molybdenium-99 (Mo-99). Mo-99 is produced by the fast neutron capture of Mo-98, and decays into Technetium-99, which is the mostly widely used radioisotope in the world. We challenge you to create a business case for the reactor that you are optimizing by predicting cash flows related to the sale of Mo-99. This is an interseting problem for a couple reasons:
- It is coupled to your optimization
- The production capacity of your reactor would likely have a large impact on the global production rate of Mo-99, and your pricing model needs to account for this

Your team must complete the following:
1. Market Analysis
- Determine the market equilibrium price and quantity using the justified supply and demand functions; we offer two models as a starting point
- Estimate how market equilibrium changes after the new facility enters the market.
- Determine the market price your facility can expect to receive.
2. Production and Revenue Estimation
- Estimate annual Mo-99 output based on reactor capacity.
- Determine the number of units sold each year (subject to capacity constraints).
- Calculate annual revenue projections.
3. Financial Modeling
- Build a 10-year operating and financial model including:
- Revenue forecast
- Operating costs
- Depreciation
- Taxes
- Free cash flow
4. Investment Valuation
- Calculate Net Present Value (NPV) using discounted cash flow analysis (Estimate terminal value if applicable)
- Provide a clear investment recommendation supported by quantitative evidence.


### Project Assumptions and Financial Inputs (all values in Million USD)

#### Capital Investment
The initial capital investment $i$ has two components: the reactor cost $C$, the same $C$ as determined via optimization, and the initial cost of the Mo-99 production system, which you must set and justify. 

As a starting point, we predict that this should total to around $1-5 B. 

#### Molybdenum-99 Production
The neutrons from the fusion reactor must be split between usage for tritium production (which your reactor needs for fuelling) and Mo-99 production. The fraction f is up to you, although the tritium production rate must, of course, exceed the reaction rate of your reactor. We use a (very optimistic) tritium breeding ratio (TBR, i.e. number of tritium atoms produced per fusion event). 

$$\text{TBR} = 1.5-1.7$$
$$Q_T = (1-f) × R_a × \text{TBR} \text{ (Tritium production rate)}$$
$$Q_{Mo99} = η × R_a × f \text{ (Mo-99 production rate)}$$

Where:
- Q_{Mo99} = annual Mo-99 production (curies/year)
- η = conversion efficiency
> Hint: You will have to research how efficient you'd expect your Mo-99 production to be
- R_a = annual neutron flux
> Hint: $R_a$ should definitely be less than your machine's actual neutron production rate $R$ as computed at the optimization stage, because your reactor needs downtime for maintenance, etc. Justify your value!

Note: Tc-99 has a 6-day lifespan.

#### COGS Function

*Project Parameters*
- Project Life: Base case = 10 years (longer life may be justified)
- Discount Rate (WACC): 10%
- Corporate Tax Rate: 25%
- Depreciation: Per unit depreciation over over lifespan (First must estimate total capability of reactor)
- Capacity Assumption: justified by you
- Fixed Overhead Costs: $1 million annually
- COGS: Mo-98 costs, electricity consumed by the reactor for heating
- Working Capital: 10% of annual revenue (recovered in final year)
- Market Demand Growth: 5% annually

#### Market Structure - base case
We offer the following linear supply and demand model as a starting point for pricing the Mo-99. Feel free to consider other pricing models and justify them; extra credit will be given. 

*Demand Function*
$$Q_d=350,000 − 100P$$

Where:
- $Q_d$ = demand (curies/year)
- $P$ = price per unit (in thousands of dollars)

*Total Supply function*
$$Q = 600,000 +Q_{Mo99} + 115P $$
Where:
- $Q$ = supply of Mo-99 (curies/year)
- $Q_{Mo99}$ = Your reactor's annual production rate of Mo-99
- $P$ = price per Ci of Mo99 (in thousands of dollars)

Note that the current market price of Mo-99 is $1,500 per curie. 


